# Evaluator Module
The Evaluator module creates evaluation reports.

Reports contain evaluation metrics depending on models specified in the evaluation config.

In [35]:
# reloads modules automatically before entering the execution of code
%reload_ext autoreload
%autoreload 2

# third parties imports
import numpy as np 
import pandas as pd
# -- add new imports here --

# local imports
from configs import EvalConfig
from constants import Constant as C
from loaders import export_evaluation_report
from loaders import load_ratings
import models as M

# -- add new imports here --

from surprise.model_selection import train_test_split, LeaveOneOut
from surprise import accuracy


# 1. Model validation functions
Validation functions are a way to perform crossvalidation on recommender system models. 

In [36]:
data = load_ratings(surprise_format=True)
print(data) #validation of the surprise format

In [37]:
def generate_split_predictions(algo, ratings_dataset, eval_config):
    """Generate predictions on a random test set specified in eval_config
        Inputs :
            algo is a surprise algorithm
            ratings_datasets is a surprise Datasets
            eval_config is the evaluator

        Outputs : prediction is a list of (userID (uid), moviesID (iid), rating (r_ui), estimated_rating (est), details (for impossible predictions))
         "" [
                (user_A, movie_1, 4.5),
                (user_B, movie_1, 3.2),
                (user_A, movie_2, 3.8),
                ...
            ]""
    """
    
    # split train/test set ()
    trainset, testset = train_test_split(
        ratings_dataset, 
        test_size= eval_config.test_size, # parameters from configs.py file set to 0.25 
        random_state=42 # Random_state to create reproducible results
    )

    algo.fit(trainset) # training

    predictions = algo.test(testset) #test

    return predictions


def generate_loo_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user on a random Leave-one-out split (LOO)
    
        Training set is a percentage of ratings_datasets
        Test set is not used only in the return

        Output : anti_testset_top_n is a dict where keys are user ids and values are list of tuples
    """
    
    loo = LeaveOneOut(n_splits=1, random_state=42) # 1 split to set aside exactly one rating per user for testing

    for trainset, testset in loo.split(ratings_dataset):

        algo.fit(trainset) #train algo on a trainset not all ratings_datasets

        anti_testset = trainset.build_anti_testset() # build anti_testset

        predictions = algo.test(anti_testset)

        anti_testset_top_n = M.get_top_n(predictions, n=eval_config.top_n_value) # eval_config.top_n_value parameters from configs.py file set to 40

        return anti_testset_top_n, testset


def generate_full_top_n(algo, ratings_dataset, eval_config):
    
    full_trainset = ratings_dataset.build_full_trainset()

    algo.fit(full_trainset)

    anti_testset = full_trainset.build_anti_testset()

    predictions = algo.test(anti_testset)

    anti_testset_top_n = M.get_top_n(predictions, n=eval_config.top_n_value) # eval_config.top_n_value parameters from configs.py file set to 40

    return anti_testset_top_n


def precompute_information(df_ratings): 
    # prepocessing method before cross-validation as no test sets are used in this type of evaluation
    # as precompute_information is used for the novelty analysis (full mode)

    """ Returns a dictionary that precomputes relevant information for evaluating in full mode
    
    Dictionary keys:
    - precomputed_dict["item_to_rank"] : contains a dictionary mapping movie ids to rankings
    - (-- for your project, add other relevant information here -- )
    """

    popularity_counts = df_ratings[C.ITEM_ID_COL].value_counts() # number of ratings given per movieID
    #print(popularity_counts)

    item_to_rank ={
        str(movieID) : str(rank) for movieID, rank in popularity_counts.rank(ascending=False, method='first').items()
        # rank(method='first') gives 1 to the movie with the most ratings, 2 to the next one, etc
    }

    precomputed_dict = {}
    precomputed_dict["item_to_rank"] = item_to_rank
    return precomputed_dict                


def create_evaluation_report(eval_config, sp_ratings, precomputed_dict, available_metrics):
    """ Create a DataFrame evaluating various models on metrics specified in an evaluation config.  
    """
    evaluation_dict = {}
    for model_name, model, arguments in eval_config.models:
        print(f'Handling model {model_name}')
        algo = model(**arguments)
        evaluation_dict[model_name] = {}
        
        # Type 1 : split evaluations
        if len(eval_config.split_metrics) > 0:
            print('Training split predictions')
            predictions = generate_split_predictions(algo, sp_ratings, eval_config)
            for metric in eval_config.split_metrics:
                print(f'- computing metric {metric}')
                assert metric in available_metrics['split']
                evaluation_function, parameters =  available_metrics["split"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(predictions, **parameters) 

        # Type 2 : loo evaluations
        if len(eval_config.loo_metrics) > 0:
            print('Training loo predictions')
            anti_testset_top_n, testset = generate_loo_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.loo_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters =  available_metrics["loo"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)
        
        # Type 3 : full evaluations
        if len(eval_config.full_metrics) > 0:
            print('Training full predictions')
            anti_testset_top_n = generate_full_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.full_metrics:
                assert metric in available_metrics['full']
                evaluation_function, parameters =  available_metrics["full"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(
                    anti_testset_top_n,
                    **precomputed_dict,
                    **parameters
                )
        
    return pd.DataFrame.from_dict(evaluation_dict).T

## 1.1. Test Functions

In [38]:
data = load_ratings(surprise_format=True)
config = EvalConfig()

In [39]:

#Test Baseline 1 (return 2)
algo_cst = models.ModelBaseline1()

cst_split = generate_split_predictions(algo_cst, data, config)

print(f"3 first prediction (Split) : {cst_split[:3]}")

top_n_dict_loo, testset_loo = generate_loo_top_n(algo_cst, data, config)

first_user_loo = list(top_n_dict_loo.keys())[0]
print(f" Top-N LOO for user {first_user_loo} : {top_n_dict_loo[first_user_loo][:3]}")

top_n_dict_full = generate_full_top_n(algo_cst, data,config)

first_user_full = list(top_n_dict_full.keys())[0]
print(f" Top-N Full for user {first_user_full} : {top_n_dict_full[first_user_full][:3]}")

print("Estimation for ModelBaseline 1 is always 2")


3 first prediction (Split) : [Prediction(uid=646, iid=2012, r_ui=5.0, est=2, details={'was_impossible': False}), Prediction(uid=652, iid=100017, r_ui=3.0, est=2, details={'was_impossible': False}), Prediction(uid=30, iid=2871, r_ui=4.0, est=2, details={'was_impossible': False})]
 Top-N LOO for user 1 : [(2761, 2), (7285, 2), (899, 2)]
 Top-N Full for user 1 : [(69122, 2), (52328, 2), (37384, 2)]
Estimation for ModelBaseline 1 is always 2


In [40]:
# Test Baseline 4 (SVD prediction)
algo_SVD = models.ModelBaseline4()

svd_split = generate_split_predictions(algo_SVD, data, config)

print(f"3 first prediction (Split) : {svd_split[:3]}")

top_n_dict_loo, testset_loo = generate_loo_top_n(algo_SVD, data, config)

first_user_loo = list(top_n_dict_loo.keys())[0]
print(f" Top-N LOO for user {first_user_loo} : {top_n_dict_loo[first_user_loo][:3]}")

top_n_dict_full = generate_full_top_n(algo_SVD, data,config)

first_user_full = list(top_n_dict_full.keys())[0]
print(f" Top-N Full for user {first_user_full} : {top_n_dict_full[first_user_full][:3]}")


3 first prediction (Split) : [Prediction(uid=646, iid=2012, r_ui=5.0, est=3.713799771771252, details={'was_impossible': False}), Prediction(uid=652, iid=100017, r_ui=3.0, est=4.201770598857985, details={'was_impossible': False}), Prediction(uid=30, iid=2871, r_ui=4.0, est=4.456673571679892, details={'was_impossible': False})]
 Top-N LOO for user 1 : [(745, 3.6557617909233424), (912, 3.6531186699211), (318, 3.634983271848797)]
 Top-N Full for user 1 : [(1203, 3.7086564932281143), (905, 3.683069145145117), (926, 3.673956175907538)]


# 2. Evaluation metrics
Implement evaluation metrics for either rating predictions (split metrics) or for top-n recommendations (loo metric, full metric)

In [41]:
def get_hit_rate(anti_testset_top_n, testset): #LeaveOnOut (loo metrics)
    """Compute the average hit over the users (loo metric)
    
    A hit (1) happens when the movie in the testset has been picked by the top-n recommender
    A fail (0) happens when the movie in the testset has not been picked by the top-n recommender

    Returns : hit_rate a float (betwenn 0 and 1)
    """
    
    hits = 0
    total_users = len(testset)

    if total_users ==0 : return 0

    for uid, iid, _ in testset:
        if uid in anti_testset_top_n:
            user_recos = {movies_id for (movies_id, rating) in anti_testset_top_n[uid]}

            if iid in user_recos:
                hits += 1
    
    hit_rate = hits/total_users

    return hit_rate


def get_novelty(anti_testset_top_n, item_to_rank): 
    
    # linked with precompute_information that ranked the most famous movies
    """Compute the average novelty of the top-n recommendation over the users (full metric)
    
    The novelty is defined as the average ranking of the movies recommended
    """
    total_novelty_sum = 0
    user_count = len(anti_testset_top_n)

    if user_count == 0:
        return 0

    for uid, user_recos in anti_testset_top_n.items():
        # Sum rank for the recommended movieId to this user
        # .get(str(movieID), max_rank) is used as a security if a movieID is not in the dict
        max_rank = len(item_to_rank)
        current_user_novelty = sum([item_to_rank.get(str(movieID), max_rank) for movieID, _ in user_recos]) 
        # user_recos are the recommendation given to a userid. It is display as a tuple (movieID,rating) 
        # the rating is not used in the novelty evaluation (need to recommend unpopular movie)
        # higher novelty means less popular movies as a blockbuster is very popular rank is very low

        total_novelty_sum += current_user_novelty

    average_rank_sum = total_novelty_sum / user_count

    return average_rank_sum

# 3. Evaluation workflow
Load data, evaluate models and save the experimental outcomes

In [42]:
# 1. Chargement des données au format Surprise
sp_data = load_ratings(surprise_format=True)

# 2. Préparation des métriques disponibles
available_metrics = {
    "split": {"rmse": (accuracy.rmse, {"verbose": False})},
    "loo": {"hit_rate": (get_hit_rate, {})},
    "full": {"novelty": (get_novelty, {})}
}

# 3. Données pré-calculées (Popularité pour la Novelty)
df_raw = load_ratings(surprise_format=False)
popularity = df_raw[C.ITEM_ID_COL].value_counts()
item_to_rank = {iid: rank for rank, (iid, count) in enumerate(popularity.items(), 1)}
precomputed_dict = {"item_to_rank": item_to_rank}

# 4. Génération du DataFrame final
final_report = create_evaluation_report(
    EvalConfig(), 
    sp_data, 
    precomputed_dict, 
    available_metrics
)

# 5. Exportation (Point 6)
export_evaluation_report(final_report)

# Affichage du tableau
final_report

Handling model baseline_1
Training split predictions
- computing metric rmse
Training loo predictions
Training full predictions
Handling model baseline_2
Training split predictions
- computing metric rmse
Training loo predictions
Training full predictions
Handling model baseline_3
Training split predictions
- computing metric rmse
Training loo predictions
Training full predictions
Handling model SVD
Training split predictions
- computing metric rmse
Training loo predictions
Training full predictions
Rapport exporté avec succès : report_2026_04_15.csv


,rmse,hit_rate,novelty
baseline_1,1.866449,0.00149,362640.0
baseline_2,1.846532,0.00000,362640.0
baseline_3,1.063200,0.00149,362640.0
SVD,0.909210,0.04769,362640.0


In [43]:
AVAILABLE_METRICS = {
    "split": {
        "mae": (accuracy.mae, {'verbose': False}), # absolute error non-negative, less sensitive to outliers as RMSE get this (error)²
        # -- add new split metrics here --
        "rmse": (accuracy.rmse, {'verbose': False}), # root (squared error) : error < 1 are negligible (getter even lower) but error >1 are getting higher
        'mse':(accuracy.mse, {'verbose' : False}), # squared error : as rmse but without root so the error are very high not between 0.5 & 5.
        'fcp':(accuracy.fcp, {'verbose' : False}), # doesn't look at rating but the ranking in correct order (fcp : Fraction of Concordance Pairs)
        
        
        # Note : if a metric is added it also needs to be done in split_metrics= [] in configs.py 
    },
    # -- add new types of metrics here --
}

sp_ratings = load_ratings(surprise_format=True)
precomputed_dict = precompute_information()
evaluation_report = create_evaluation_report(EvalConfig, sp_ratings, precomputed_dict, AVAILABLE_METRICS)
export_evaluation_report(evaluation_report)

TypeError: precompute_information() missing 1 required positional argument: 'df_ratings'